# Hyperparameter Tuning + Feature Engineering: Grid Search, Pipelines, and Leakage Detection

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb09_tuning_feature_engineering_project_baseline_instructor.ipynb)

---

> **🎯 nb09 = the toolkit closer.** By the end of today, you will hold the full mid-course toolkit — every concept (bias–variance, overfitting vs underfitting, leakage, regression vs classification, the curse of dimensionality), every sklearn primitive (`Pipeline`, `ColumnTransformer`, `Ridge`/`Lasso`/`LogisticRegression`, `cross_val_score`, `GridSearchCV`/`RandomizedSearchCV`), and every decision rule (CI-overlap, "fit inside the pipeline", "test set stays locked"). **Section 3 of today's notebook is your one-page consolidated recap** of everything nb01–nb09 has built — keep it close as a reference for any future analysis.

---


## Learning Objectives

By the end of this notebook, you will be able to:

1. Run `GridSearchCV` and `RandomizedSearchCV` on known models and read `cv_results_` as a table of nb08-style CV runs
2. Apply the 95% confidence-interval overlap rule from nb08 to pick the simplest model among the top candidates in `cv_results_`
3. Build a `ColumnTransformer` that handles both categorical and numeric features inside a single `Pipeline`
4. Use `FunctionTransformer` to embed domain-specific feature engineering (ratios, bins) inside the pipeline without leakage
5. Detect a data-leakage bug in a provided pipeline by comparing CV scores before and after the fix
6. Explain why every feature-engineering step must live inside the pipeline that `cross_val_score` or `GridSearchCV` evaluates

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**, one at the end of each big section. Please complete them before submitting your notebook.

---

## 💼 Why This Matters: From One Knob to a Dial (and a Leaky Pipeline to Fix)

Two business cases are on your desk today — one returning from earlier notebooks, one brand new.

**Section A — HomeValue Analytics + MedScreen.** In nb05 the CFO signed off on a Ridge baseline with a single hand-picked $\alpha = 1.0$. Then nb08 taught you that any "single number" claim deserves a 95% confidence interval. The natural next question is the one the CFO actually wanted to ask: *"is $\alpha = 1.0$ really the best choice, or did you just pick a round number?"* The tool that answers this is `GridSearchCV` — and you will see that it is nb08's CV ritual run once per candidate value of $\alpha$, producing a ranked table you already know how to read. The Health Department wants exactly the same treatment for MedScreen's logistic regression — a principled $C$-grid search instead of one hand-chosen value.

**Section B — TechCorp Talent Analytics.** You have been loaned to TechCorp, a 1,500-person SaaS company. The People Analytics team wants to flag employees at high risk of resigning in the next 6 months so HR can open retention conversations early. The economics are sharp: losing a mid-level engineer costs about USD 75,000 (recruitment + 6 months of productivity loss + ramp time); a retention conversation costs about USD 500 (manager time + modest retention bonus). Even a modest lift over "managers guessing" pays for itself many times over.

TechCorp's dataset has something California Housing and Breast Cancer never had: **real categorical columns** — `department`, `job_level`, `remote_status`, `manager_id`. That means you finally need `ColumnTransformer` to mix one-hot encoding with standardization in one pipeline. And while you were away, an intern shipped a first draft of the modeling pipeline. Their CV ROC-AUC looked suspiciously good… because it is. You will spot the leak, fix it, watch the inflated score collapse, and then find a second, subtler leak on your own.

By the end of today you own a workflow that plugs directly into your Kaggle competition submission: **ColumnTransformer → FunctionTransformer → GridSearchCV → `cv_results_` ranked by CI overlap → one final pipeline refit on all training data**. If that sentence feels abstract now, it will feel like a muscle memory by the end of the notebook.

> **Today's focus:** turn nb08's single CV run into a *ranked table* of CV runs, learn to read that table with the CI-overlap rule, and build the first pipeline in this course that handles real categorical data — all while staging two concrete leakage case studies so that "don't leak" stops being a lecture warning and becomes something you can spot in the wild.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

print('Setup complete.')
print(f'Random seed: {RANDOM_SEED}')

**Reading the output:**

The setup cell imports today's full toolbox in one shot, so nothing later has to stop and re-import. Here is a quick tour of what each piece does and where you will meet it.

The **two tuners** are `GridSearchCV` and `RandomizedSearchCV`. Both take a pipeline plus a parameter grid (or distribution), run 5-fold CV for every candidate configuration, and store every fold's score in a table called `cv_results_`. Think of `cv_results_` as "a stack of nb08 runs" — every row is one complete nb08 evaluation with its own mean, SD, and potential 95% CI. `GridSearchCV` enumerates every combination in a fixed grid; `RandomizedSearchCV` samples from distributions when the grid would be too big to enumerate.

**`ColumnTransformer`** is the sklearn idiom for *"different preprocessing per column type."* Today it lets you apply `StandardScaler` to the numeric columns and `OneHotEncoder(handle_unknown='ignore')` to the categorical columns inside a single `Pipeline`. `handle_unknown='ignore'` is a safety net: if a prediction-time row contains a category never seen in training, the encoder outputs zeros for that row's category block instead of crashing.

**`FunctionTransformer`** wraps a plain Python function — e.g., "compute a ratio column" — into a pipeline step so that the function refits correctly on each CV fold. You will use it in Section 2 for a domain feature that the HR Business Partner asks about.

**`SelectKBest`** picks the top-$k$ features by a univariate score (today, `f_classif`). It is genuinely useful — *and* it is the transform you will misuse on purpose in PAUSE-AND-DO 2 to see how leakage feels when you accidentally set it up wrong.

**`scipy.stats`** is here for one call: `stats.t.ppf(0.975, df=k-1)`, the Student's $t$ critical value from nb08. The CI-overlap rule drives both exercises today.

`RANDOM_SEED = 474` is inherited from nb01–nb08, so every fold is directly comparable to what you computed in prior notebooks.

> **A question that often comes up here:** *"Do I need to know exactly when to reach for `GridSearchCV` versus `RandomizedSearchCV`?"* Simple rule: if the grid is under \~100 combinations, use `GridSearchCV` — exhaustive is cheap. If the grid explodes (imagine 4 learning rates × 4 depths × 4 leaf sizes × 4 subsample rates = 256), use `RandomizedSearchCV` with `n_iter=30` or so. Same output format, a fraction of the compute.

---

## 1. Section A — Grid Search as nb08 × a Grid

### 1.1 From one CV run to many

Take a breath, because this section is a bigger conceptual step than it looks. Today you are going to run dozens of models instead of one — but the *mechanics* are exactly what you already know.

**Start with what you have.** In nb08 Section 2 you ran a single 5-fold CV on one Ridge pipeline with $\alpha = 1.0$ and got back five fold scores. You averaged them (mean), measured their spread (SD), and built a 95% confidence interval (CI) using Student's $t$. That gave you a defensible answer to the question *"is my single validation score representative?"* — one pipeline, one CI, one decision.

**Now the natural next question.** You only tried $\alpha = 1.0$ because nb05 suggested it as a reasonable starting point. But is it really the best choice? What if $\alpha = 10$ or $\alpha = 100$ would do better? The only honest way to answer is to **run NB08's 5-fold CV ritual on every candidate $\alpha$** and look at the resulting distribution of CIs.

That loop is what `GridSearchCV` automates. You hand it three things:

1. A **pipeline** (e.g., `StandardScaler + Ridge`).
2. A **parameter grid** — a dictionary of candidate values: `{'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}`.
3. A **CV splitter** (the same `KFold(5)` or `StratifiedKFold(5)` you know).

Internally `GridSearchCV` does exactly what you would have done by hand — fits 6 α × 5 folds = 30 separate models, scores each one, averages the folds, and stacks the results into a DataFrame called `cv_results_`. **Every row of `cv_results_` is one complete nb08 run**, with its own mean, SD, and (with two lines of `scipy.stats`) its own 95% CI. The only thing new today is learning to read that table and pick a winner from it.

> **An analogy that often helps here.** Imagine nb08's 5-fold CV as a single lab experiment — one recipe, five beakers, one result. `GridSearchCV` is the whole lab running six experiments in parallel, each with a different recipe variation, and writing up all six results as a table on the whiteboard. Your job is to read the whiteboard and pick the recipe worth keeping — using the CI-overlap rule from nb08, not just the highest mean.

### The professional workflow in four steps

Whenever you run grid search from here forward — for Ridge α today, for C in logistic regression later, for tree depth and learning rate in Weeks 3 and 4 — follow this four-step pattern:

1. **Rank `cv_results_` by `mean_test_score`.** Sklearn writes one row per candidate; sorting puts the highest-mean candidate at the top.
2. **Read the top 3–5 rows** and compute each one's 95% CI using the Student's $t$ formula from nb08 (half-width = $t_{0.975,\, 4}$ × SD / √k, with $t \approx 2.776$ at $k=5$).
3. **Apply the CI-overlap rule.** If the top row's CI overlaps the second row's CI, the two are statistically tied. Pick the *simpler* of the two — stronger regularization on a Ridge grid, smaller $C$ on a LogReg grid.
4. **Refit the chosen pipeline on all training data** and lock it. The test set stays untouched for nb14's ceremony.

This protocol is the spine of every model-selection decision in the rest of the course. If you can read `cv_results_` and apply the CI-overlap rule, you can evaluate any tuning library you will ever use — sklearn, Optuna, Ray Tune, you name it. They all produce the same kind of table.

> **A question that often comes up right here:** *"If `best_params_` already picks the winner, why do I need to inspect the top rows by hand?"* Because `best_params_` is the best *mean*, not the best *supported* choice. A 0.0005 difference in mean is inside the fold-to-fold noise (you saw that in nb08 — the CI half-width at $k=5$ is roughly 1.24 × SD, not a tight 2 × SD / √5). The CI-overlap rule asks *"are these two candidates statistically distinguishable?"* — and when they are not, you should pick on simplicity, interpretability, or runtime, not on a coin-flip win in the fifth decimal place.

---

### 1.2 GridSearchCV on Ridge — HomeValue Analytics

Load California Housing, apply the same 60/20/20 split used in nb01–nb08, and sweep Ridge over a 6-value $\alpha$ grid.

> 💡 **Gemini Prompt:** "Load the California Housing dataset. Do a 60/20/20 train/val/test split with `random_state=RANDOM_SEED`. Build a pipeline with `StandardScaler` + `Ridge`. Run `GridSearchCV` with `param_grid={'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}`, `cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)`, `scoring='r2'`, and `return_train_score=False`. After fitting, print `grid.best_params_` and `grid.best_score_`. Then build a DataFrame from `grid.cv_results_` with columns `param_ridge__alpha`, `mean_test_score`, `std_test_score`, `rank_test_score`, sort by rank, and print the full table."
>
> **After running, verify:**
> - The grid fits 6 configurations × 5 folds = 30 sub-fits
> - `best_params_` is printed with the winning $\alpha$
> - The `cv_results_` DataFrame has 6 rows, one per $\alpha$
> - `rank_test_score` starts at 1 for the best row


In [ ]:
# --- Load California Housing and apply the 60/20/20 split ---
cal = fetch_california_housing(as_frame=True)
X_reg = cal.data
y_reg = cal.target

X_reg_temp, X_reg_test, y_reg_temp, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_reg_train, X_reg_val, y_reg_train, y_reg_val = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
print(f'Train: {len(X_reg_train)} | Val: {len(X_reg_val)} | Test: {len(X_reg_test)} (locked)')

# --- Build pipeline + grid ---
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=RANDOM_SEED))
])

alpha_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

grid_ridge = GridSearchCV(
    ridge_pipeline,
    param_grid=alpha_grid,
    cv=cv_reg,
    scoring='r2',
    return_train_score=False,
    n_jobs=-1
)
grid_ridge.fit(X_reg_train, y_reg_train)

print(f'\nBest params: {grid_ridge.best_params_}')
print(f'Best 5-fold CV R^2 (mean): {grid_ridge.best_score_:.4f}')

# --- cv_results_ as a ranked table ---
results_ridge = (
    pd.DataFrame(grid_ridge.cv_results_)
    [['param_ridge__alpha', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .reset_index(drop=True)
)
print('\nRanked grid results:')
print(results_ridge)

**Reading the output:**

Three artifacts land on screen, and you should walk through them in order.

**First: `grid.best_params_`**. This prints the single winner by mean — the $\alpha$ value whose 5-fold CV $R^2$ was highest on average across the five folds. Sklearn's default behavior picks this row automatically; it is the answer the framework gives you without asking you to think. Treat it as a *starting point*, not the final decision.

**Second: `grid.best_score_`**. This is the mean CV $R^2$ of that winning row. It tells you the headline number but *not* the spread — no SD, no CI. Reporting `best_score_` on its own is the thing nb08 warned you against: a number without its uncertainty is not a decision-ready number.

**Third: the sorted `cv_results_` table**. This is where the actual decision gets made. The columns you care about:

- **`param_ridge__alpha`** — the candidate $\alpha$ for that row. (Aside: the double underscore `ridge__alpha` is sklearn's way of reaching into a named pipeline step. Your pipeline had a step called `'ridge'`, and that step's parameter is `alpha`, so sklearn glues them with `__`. If you had named the step `'model'`, you would write `model__alpha` in the grid.)
- **`mean_test_score`** — the mean 5-fold CV $R^2$ for this candidate. This is the "mean" from nb08's vocabulary.
- **`std_test_score`** — the sample standard deviation of the five fold scores. This is the "SD" from nb08's vocabulary. Combined with $k$ and $t_{0.975}$ it gives you the 95% CI.
- **`rank_test_score`** — 1 for the candidate with the best mean, 2 for second-best, etc. Sorting by this puts the winner at the top.

**What to look at first, second, third.** (1) Read `best_params_` to see which $\alpha$ won by mean. (2) Eyeball the top two or three rows in the sorted table — are their `mean_test_score` values close (within a few thousandths)? (3) Glance at `std_test_score` — if the SDs are on the same order of magnitude as the mean differences, the winner is fragile. If the top candidates' means differ by less than roughly 1.24 × SD / √k, their CIs almost certainly overlap, and you should not be picking on mean alone.

**But the mean is not the decision.** The next cell converts every row's SD into a proper 95% CI (Student's $t$, $k-1$ degrees of freedom) and tests whether the top-ranked candidate's CI overlaps the runner-up's. That is the cell that actually picks the winner.

> **A question that often comes up here:** *"Will the winning $\alpha$ be stable if I change `RANDOM_SEED`?"* Often not — which is exactly why the CI-overlap rule exists. A seed-stable winner has a CI that clearly clears the runner-up's CI; its lead is real. A seed-fragile "winner" has a CI that overlaps everyone in the top few rows; reshuffle the seed and the "winner" flips. Reporting the first as *"we chose $\alpha = 10$ because its CI does not overlap the runner-up"* is a defensible claim. Reporting the second as *"we chose $\alpha = 10$ because it had the highest mean"* is brittle — the next data scientist to inherit your code will reshuffle the seed and get a different answer.

---

### 1.3 CI-overlap rule on `cv_results_`

You already know how to build a 95% CI — you built one in nb08 Section 2 from the five fold scores of a single run. The only new trick here is doing it **in bulk**, once per row of `cv_results_`.

**What sklearn hands you.** `GridSearchCV` stores every fold's score in columns called `split0_test_score`, `split1_test_score`, …, `split4_test_score`. Each of those columns has one value per candidate — the $R^2$ that candidate got on fold $i$. If you select the five `splitN_test_score` columns for one row, you have exactly the five fold scores that nb08's CI formula expects.

**What you do with them.** For every row:

1. Take the five fold scores.
2. Compute the mean — that is `mean_test_score`, already in the DataFrame.
3. Compute the sample SD with `ddof=1` (the same `.std(ddof=1)` from nb08 — unbiased, divides by $k-1$).
4. Compute the half-width as $t_{0.975,\,k-1} \times \mathrm{SD} / \sqrt{k}$. At $k=5$, $t_{0.975,\,4} \approx 2.776$, so the half-width works out to roughly $1.24 \times \mathrm{SD}$.
5. The 95% CI is `[mean - half_width, mean + half_width]`.

**Apply the overlap test to the top two rows.** After sorting by mean, pull the top-1 and top-2 rows. Two CIs overlap if *any* height is shared — formally, if `top1.ci_high >= top2.ci_low` AND `top2.ci_high >= top1.ci_low`. Visually: if the two error bars touch at any point, they overlap.

**The decision, stated once more so it sticks.** If the top two CIs overlap, the two candidates are statistically tied on this data. Pick the simpler (more regularized) one. If they do not overlap, the top-ranked candidate wins outright — its advantage is real.

This is the exact same decision rule you practiced in nb08 Exercise 1 (Ridge vs. plain OLS) and nb08 Exercise 2 (LogReg $C=1.0$ vs. $C=0.01$). All that has changed is that now you are applying it across a *grid* of candidates, not just between two named models.

In [ ]:
# --- Compute per-row 95% CI from fold scores ---
k = 5
t_crit = stats.t.ppf(0.975, df=k - 1)

fold_cols = [f'split{i}_test_score' for i in range(k)]
cvr = pd.DataFrame(grid_ridge.cv_results_).copy()
cvr['mean'] = cvr[fold_cols].mean(axis=1)
cvr['sd'] = cvr[fold_cols].std(axis=1, ddof=1)
cvr['half_w'] = t_crit * cvr['sd'] / np.sqrt(k)
cvr['ci_low'] = cvr['mean'] - cvr['half_w']
cvr['ci_high'] = cvr['mean'] + cvr['half_w']
cvr = cvr.sort_values('mean', ascending=False).reset_index(drop=True)

print('Top candidates by mean 5-fold CV R^2 (with 95% CI):\n')
print(cvr[['param_ridge__alpha', 'mean', 'sd', 'ci_low', 'ci_high']].head())

# --- Plot top 6 with CI error bars ---
fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"α={a}" for a in cvr['param_ridge__alpha']]
ax.bar(labels, cvr['mean'], yerr=cvr['half_w'], color='steelblue',
       capsize=8, edgecolor='black')
ax.set_ylabel('5-fold CV R^2')
ax.set_title('Ridge α grid — mean CV R^2 with 95% CI (ranked)',
             fontsize=12, fontweight='bold')
for i, (m, h) in enumerate(zip(cvr['mean'], cvr['half_w'])):
    ax.text(i, m + h + 0.003, f'{m:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# --- CI-overlap between top-1 and top-2 ---
top1, top2 = cvr.iloc[0], cvr.iloc[1]
overlap = not (top1['ci_high'] < top2['ci_low'] or top2['ci_high'] < top1['ci_low'])
if overlap:
    print(f"\n→ Top-1 (α={top1['param_ridge__alpha']}) and Top-2 (α={top2['param_ridge__alpha']}) 95% CIs OVERLAP.")
    simpler = max(top1['param_ridge__alpha'], top2['param_ridge__alpha'])
    print(f'  Statistical tie — pick the simpler (larger α): α={simpler}.')
else:
    print(f"\n→ Top-1 (α={top1['param_ridge__alpha']}) CI does NOT overlap Top-2. "
          f"Top-1 wins outright.")

**Reading the output:**

Two things to read carefully: the printed table and the bar plot.

**The printed table** has one row per $\alpha$ candidate, sorted by mean (best at the top). Columns to scan: `param_ridge__alpha` (which $\alpha$), `mean` (CV mean $R^2$), `sd` (fold-to-fold standard deviation), and the two CI endpoints `ci_low` and `ci_high`. A useful habit: check whether the top candidate's `ci_low` is above every other candidate's `ci_high`. If yes, the top wins outright; if not, at least one other candidate is statistically indistinguishable.

**The bar plot** encodes the same information visually. Each bar is one candidate — its height is the `mean_test_score`, and the vertical error bars extending above and below the bar mark the 95% CI (`ci_low` at the bottom, `ci_high` at the top). Bars are ordered by rank, left to right, so the highest-mean candidate is leftmost.

Now do the eyeball test the CI-overlap rule is asking you to do. **Does the top bar's CI (the whiskers) clear the runner-up's CI?** Two ways this can come out:

- **They do not overlap** — the top bar's lower whisker sits above the second bar's upper whisker, with a visible gap. The top candidate is a real winner; its mean difference over the runner-up is bigger than the noise can explain.
- **They overlap at any height** — the whiskers share a range, even if the means are different. The two candidates are statistically tied on this data. The professional move is to pick the *more regularized* (larger $\alpha$) of the two — stronger regularization means a simpler model, which generalizes more reliably to new hospitals / new customer cohorts / new months of data.

On California Housing with `RANDOM_SEED = 474`, the top three α values usually overlap almost completely. `grid.best_params_` picks one of them by the tiniest of margins on the mean, but the CI-overlap rule says the top three are all defensible choices and the simplest (largest α) is the professional pick.

> **A question that often comes up here:** *"What counts as 'overlapping' — do the whiskers have to touch exactly, or is it anywhere?"* Any shared range at all. If top-1's `ci_low` is 0.59 and top-2's `ci_high` is 0.60, they overlap at \~0.595 and you are in a tie. If top-1's `ci_low` is 0.60 and top-2's `ci_high` is 0.59, they do **not** overlap — top-1's worst-case is better than top-2's best-case. When you report this to a stakeholder, show the bar plot with the CI whiskers so the overlap (or lack of it) is visible at a glance.

---

### 1.4 RandomizedSearchCV — when the grid is too big

A grid of 4 `C` values × 3 `penalty` × 2 `solver` is already 24 fits × 5 folds = 120 sub-fits. Realistic grids for gradient boosting or neural nets balloon into the thousands. `RandomizedSearchCV` samples `n_iter` points from distributions instead of enumerating every combination — same output format (`cv_results_`, `best_params_`, `best_score_`), a fraction of the compute.

#### A quick primer on `C` — what is the parameter you are about to sweep?

Before turning the tuner loose, one paragraph on what `C` actually controls — because it is named in a way that trips up almost everyone the first time. **`C` is the *inverse* of regularization strength.** Logistic regression learns one coefficient per feature, and without any restriction it can push those coefficients arbitrarily large to fit every twist in the training data — including noise. *Regularization* is the penalty that pulls those coefficients back toward zero, trading a sliver of training fit for a lot of stability on data the model has not seen yet. `C` controls how aggressive that pull is, *inversely*: **small `C` (e.g., `0.01`) means strong regularization, small coefficients, and a simpler, smoother decision boundary that resists chasing individual training points; large `C` (e.g., `100`) means weak regularization, coefficients are free to grow, and the boundary becomes flexible enough to either capture real structure or memorize noise; `C = 1.0` is sklearn's default and a sensible neutral starting point.** The values swept throughout this notebook (`0.001, 0.01, 0.1, 1, 10, 100, 1000`) are spaced on a log scale on purpose — each step is 10× more flexible than the previous one, which is why the Gemini Prompt below uses `loguniform(1e-3, 1e3)` rather than a plain `uniform`: log-uniform sampling gives every order of magnitude an equal chance of being explored. This also locks in the CI-overlap tie-breaker with a concrete reason. When two `C` values produce statistically tied ROC-AUCs, the *smaller* `C` wins because it ships simpler coefficients — a model you can defend to a CFO or Health Department by saying *"we picked the simpler model when the evidence was tied"*, rather than fighting over a 0.0005 mean difference that will not survive a reseed.

> 💡 **Gemini Prompt:** "On Breast Cancer (same stratified 60/20/20 split as nb06/nb07/nb08), build a pipeline `StandardScaler + LogisticRegression(max_iter=5000)`. Use `RandomizedSearchCV` with `param_distributions={'clf__C': scipy.stats.loguniform(1e-3, 1e3)}`, `n_iter=20`, `cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED)`, `scoring='roc_auc'`, `random_state=RANDOM_SEED`. Print `best_params_` and `best_score_`, then show the top 5 rows of `cv_results_` sorted by `rank_test_score`."
>
> **After running, verify:**
> - 20 random `C` values were sampled (each on a log scale)
> - `best_score_` is a mean ROC-AUC between 0.98 and 1.00
> - Top-5 rows show a range of `C` values, not all clustered in one spot

In [ ]:
# --- Breast Cancer stratified 60/20/20 ---
bc = load_breast_cancer(as_frame=True)
X_clf = bc.data
y_clf = bc.target

X_clf_temp, X_clf_test, y_clf_temp, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_clf_train, X_clf_val, y_clf_train, y_clf_val = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_clf_temp
)
print(f'Train: {len(X_clf_train)} | Val: {len(X_clf_val)} | Test: {len(X_clf_test)} (locked)')

# --- Randomized search over C ---
log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])

rand_search = RandomizedSearchCV(
    log_pipeline,
    param_distributions={'clf__C': stats.loguniform(1e-3, 1e3)},
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    scoring='roc_auc',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rand_search.fit(X_clf_train, y_clf_train)

print(f'\nBest params: {rand_search.best_params_}')
print(f'Best 5-fold CV ROC-AUC (mean): {rand_search.best_score_:.4f}')

# --- Top 5 ---
top5 = (
    pd.DataFrame(rand_search.cv_results_)
    [['param_clf__C', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .head()
    .reset_index(drop=True)
)
print('\nTop 5 randomized-search candidates:')
print(top5)

**Reading the output:**

`RandomizedSearchCV` drew 20 `C` values from a log-uniform distribution between $10^{-3}$ and $10^{3}$ and ran the full 5-fold stratified CV pipeline for each one. The printed top-5 shows the winning `C` is rarely a round number — something like $C \approx 0.23$ or $C \approx 47.3$, which a fixed grid of round values would never have tried. That is the whole pitch for randomized search: you explore the space smarter than a regular grid does, at the same compute budget.

**The most important mental model to carry out of this section** is that every row of `cv_results_` — whether produced by `GridSearchCV` or `RandomizedSearchCV` — is one complete nb08 run. The tuner is a convenient loop, not a new algorithm. When you quote a score from the winner, quote it the nb08 way: *"5-fold CV ROC-AUC = 0.99 ± CI"*, never as a single magic number with no spread attached.

> **A question that often comes up here:** *"Is `RandomizedSearchCV` always better than `GridSearchCV`?"* No — for small grids (under \~50 combinations) `GridSearchCV` is both cheaper and more thorough, because it tries everything. `RandomizedSearchCV` wins when the grid gets large enough that exhaustive enumeration stops being free. A good heuristic: start with `GridSearchCV` while the grid has only one or two dimensions, and switch to `RandomizedSearchCV` the moment you add a third dimension — the total count grows multiplicatively.

---

## 📝 PAUSE-AND-DO Exercise 1 (10 minutes)

**Task:** Run `GridSearchCV` on MedScreen's Logistic Regression over an explicit `C` grid and apply the CI-overlap rule to pick a simpler winner than `best_params_` alone would give you.

This exercise is a direct application of the workflow you just saw. You will build the same `StandardScaler + LogisticRegression` pipeline used in Section 1.4, but drop `RandomizedSearchCV` for a clean `GridSearchCV` so you can eyeball every candidate on screen. Fit on `X_clf_train` / `y_clf_train` using the same `StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED)` splitter and `scoring='roc_auc'`.

Use this explicit grid:

```python
param_grid = {'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}
```

After fitting, do exactly what Section 1.3 did — compute the 95% CI on every row's `splitN_test_score` columns, sort by mean, and check whether the top-1 CI overlaps the top-2 CI. If it does, pick the *smaller* `C` (stronger regularization, simpler model) as your champion; if it does not, the top-1 wins outright. Print a one-sentence verdict.

> **Reminder on the decision rule from nb08 and Section 1.3 above:** overlapping CIs at the top of the ranking are a statistical tie. Break the tie toward the simpler model. Do not pick the one with the higher mean by 0.001 — that difference is inside the noise, and a different random seed might hand the "winner" prize to the runner-up next week.

---

> 💡 **Gemini Prompt:** "Build a Logistic Regression pipeline (`StandardScaler` + `LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)`). Using `X_clf_train`, `y_clf_train`, and `StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` with `scoring='roc_auc'`, run `GridSearchCV` over `param_grid = {'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}`. Convert `cv_results_` to a pandas DataFrame, then for every row compute the mean across the `split0_test_score`..`split4_test_score` columns, the sample SD with `ddof=1`, the half-width as `t_crit * sd / sqrt(5)` (`t_crit` is already defined in Section 1.3), and `ci_low` / `ci_high`. Sort the DataFrame by mean descending and print the columns `param_clf__C`, `mean`, `sd`, `ci_low`, `ci_high`. Compare the top-1 and top-2 95% CIs: if they overlap, set `champion_by_CI` to the *smaller* `C` of the two; otherwise set it to the top-1 `C`. Print a one-sentence verdict that names the champion and the rule that picked it."
>
> **After running, verify:**
> - A ranked DataFrame prints with columns `param_clf__C`, `mean`, `sd`, `ci_low`, `ci_high`
> - The top row has the highest mean ROC-AUC (often `C=10.0` or `C=100.0` on this data)
> - Top-1 and top-2 95% CIs visibly overlap
> - The final printed verdict says "CIs OVERLAP — pick the simpler (smaller) C" and names a `C` value ≤ 1.0 as `champion_by_CI`
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build pipeline: StandardScaler + LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)
#   2. Run GridSearchCV with param_grid={'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}
#      on X_clf_train, y_clf_train using StratifiedKFold(5, shuffle, random_state=RANDOM_SEED)
#      with scoring='roc_auc'
#   3. Extract cv_results_ as a DataFrame
#   4. For every row, compute mean / sd (ddof=1) / half_w / ci_low / ci_high across split0..split4
#   5. Sort by mean (descending)
#   6. Check whether top-1 and top-2 CIs overlap. Print the verdict.
#   7. champion_by_CI = simpler C when they overlap, otherwise top-1 C


### YOUR ANALYSIS:

**Question 1:** Which `C` has the highest mean 5-fold CV ROC-AUC?  
[Your answer]

**Question 2:** Do the top-1 and top-2 95% CIs overlap? What do you conclude?  
[Your answer]

**Question 3:** Which `C` do you recommend to the Health Department, and why? Reference the CI-overlap rule in your justification.  
[Your answer]

---

## 2. Section B — Feature Engineering, Categorical Data, and the Leakage Trap

Section A turned the CV ritual into a ranked table of candidates — one tool, one decision rule. Section B takes on a completely different problem: getting raw, messy, real-world data into a pipeline *safely*.

### 2.1 TechCorp Talent Analytics — the business case

**TechCorp** is a fictional 1,500-person SaaS company you are being loaned to for a single consulting project. Their People Analytics team has a standing ask from the CHRO (Chief Human Resources Officer): *flag employees at high risk of resigning within the next 6 months so HR can run targeted retention conversations before the employee is already out the door*. The math is crisp: losing a mid-level engineer costs the company roughly USD 75,000 (recruitment fees + 6 months of productivity loss + ramp time for the replacement); a retention conversation costs about USD 500 (manager time + modest retention bonus). Even a modest lift over *"managers guessing"* pays for itself many times over.

The People Analytics team built a dataset with the following columns:

- **Numeric features:** `years_at_company`, `satisfaction_score` (a 1–10 pulse-survey result), `salary_pct_of_market` (the company's salary as a percentage of the market median — above 100 is generous, below is underpaying), `projects_completed_last_year`, `manager_interactions_last_quarter`.
- **Low-cardinality categorical features:** `department` (5 levels — Engineering, Sales, Marketing, Operations, Support), `job_level` (5 levels — IC1, IC2, IC3, Manager, Director), `remote_status` (3 levels — onsite, hybrid, remote).
- **High-cardinality categorical:** `manager_id` (80 distinct managers, roughly 25 employees per manager). This one is going to matter a lot in Section 2.3.
- **Target:** `left_within_6mo` (binary — 1 if the employee resigned within 6 months of the snapshot, else 0).

**Why a made-up company instead of real data?** Two reasons. First, because the Kaggle competition data (which you will meet on Day 5) is the graded deliverable, we keep it in the vault for competition time. Second — and this is the *pedagogical* reason — because the machinery of Section 2 needs a dataset whose structure you control. The dramatic target-encoding leak in Section 2.3 only bites clearly when the dataset has a high-cardinality categorical with small group sizes (that is what `manager_id` with 80 levels and \~25 employees per level is engineered to produce). On a noisy real dataset the leak is often there but the signal/noise ratio can hide it. Here you will see it unambiguously.

### What makes TechCorp special in this course

TechCorp is the **first dataset you will encounter with real categorical columns**. Both California Housing (nb01–nb05) and Breast Cancer (nb06–nb08) are entirely numeric — every feature is a floating-point number. `ColumnTransformer` has been on the syllabus since nb02 but you have never actually exercised it on live categorical data because there was no categorical data to exercise it on. That changes today. Feeding a column of strings like `['Engineering', 'Sales', 'Marketing']` straight into `LogisticRegression` would raise `could not convert string to float` — you have to *encode* it first, and the professional way to encode is `OneHotEncoder(handle_unknown='ignore')` living inside a `ColumnTransformer` living inside a `Pipeline`. That three-level nesting is today's new pattern.

> **A question that often comes up here:** *"If I can encode categoricals with pandas `get_dummies`, why do I need `ColumnTransformer` and `OneHotEncoder` at all?"* You can — for exploration. For a deployable pipeline, you cannot. `pd.get_dummies` builds the columns once on the DataFrame in front of it, and if a new category appears in production data (a new department, a new manager hired after you trained), the encoded DataFrame has the wrong columns and the model silently breaks. `OneHotEncoder(handle_unknown='ignore')` inside a `Pipeline` learns the vocabulary from training data, transforms validation / test / production data consistently, and emits all-zero indicators for unseen categories instead of crashing. The discipline of "encoders live inside the Pipeline" is the same discipline of "scalers live inside the Pipeline" — if the transform looks at the data, it belongs inside the object the CV wraps.

> 💡 **Gemini Prompt:** "Generate a synthetic TechCorp attrition dataset with 2000 rows. Use `numpy.random` seeded at `RANDOM_SEED`. Build a pandas DataFrame with: numeric columns `years_at_company` (integer 0–15), `satisfaction_score` (1–10), `salary_pct_of_market` (60–140), `projects_completed_last_year` (0–12), `manager_interactions_last_quarter` (0–20); categorical columns `department` (one of Engineering/Sales/Marketing/Operations/Support), `job_level` (IC1/IC2/IC3/Manager/Director), `remote_status` (onsite/hybrid/remote), `manager_id` (80 distinct managers — one per ~25 employees). Create a binary target `left_within_6mo` whose probability is higher when `satisfaction_score` is low, `salary_pct_of_market` is low, `manager_interactions_last_quarter` is low, `remote_status == 'onsite'`, or `department == 'Support'`. Target around 22% positive rate. Also add 30 noise columns named `engagement_metric_00` through `engagement_metric_29` drawn from `rng.normal(0, 1)` — TechCorp's HRIS really does export this many engagement fields, and they matter for Section 2.3's leakage demo. Print the head, the class balance, and the dtypes so the numeric vs categorical split is obvious."


In [ ]:
# --- Generate synthetic TechCorp attrition dataset ---
rng = np.random.default_rng(RANDOM_SEED)
n = 2000

years = rng.integers(0, 16, size=n)
satisfaction = rng.integers(1, 11, size=n)
salary_pct = rng.uniform(60, 140, size=n).round(1)
projects = rng.integers(0, 13, size=n)
mgr_interactions = rng.integers(0, 21, size=n)

department = rng.choice(
    ['Engineering', 'Sales', 'Marketing', 'Operations', 'Support'],
    size=n, p=[0.35, 0.20, 0.15, 0.15, 0.15]
)
job_level = rng.choice(
    ['IC1', 'IC2', 'IC3', 'Manager', 'Director'],
    size=n, p=[0.25, 0.30, 0.25, 0.15, 0.05]
)
remote_status = rng.choice(
    ['onsite', 'hybrid', 'remote'],
    size=n, p=[0.30, 0.45, 0.25]
)

# --- High-cardinality categorical: 80 manager IDs (one manager per ~25 employees) ---
manager_id = rng.choice([f'MGR_{i:03d}' for i in range(80)], size=n)

# --- Attrition propensity ---
logit = (
    -2.0
    + 0.35 * (10 - satisfaction)
    + 0.025 * (100 - salary_pct)
    + 0.08 * (10 - mgr_interactions)
    + 0.40 * (remote_status == 'onsite').astype(float)
    + 0.35 * (department == 'Support').astype(float)
    - 0.05 * years
)
prob_leave = 1.0 / (1.0 + np.exp(-logit))
left_within_6mo = (rng.uniform(0, 1, size=n) < prob_leave).astype(int)

techcorp = pd.DataFrame({
    'years_at_company': years,
    'satisfaction_score': satisfaction,
    'salary_pct_of_market': salary_pct,
    'projects_completed_last_year': projects,
    'manager_interactions_last_quarter': mgr_interactions,
    'department': department,
    'job_level': job_level,
    'remote_status': remote_status,
    'manager_id': manager_id,
    'left_within_6mo': left_within_6mo
})

# --- 30 noise columns (TechCorp's HRIS exports many engagement telemetry metrics; most are noise) ---
# These columns are included because TechCorp's HR system really does export dozens of fields.
# They also make the selection-bias leak in Section 2.3 clearly visible.
NOISE_COLS = [f'engagement_metric_{j:02d}' for j in range(30)]
for col in NOISE_COLS:
    techcorp[col] = rng.normal(0.0, 1.0, size=n)

print('TechCorp Talent Analytics — first 5 rows (core columns):')
print(techcorp.iloc[:, :9].head())
print(f"\nShape (core + {len(NOISE_COLS)} noise engagement metrics): {techcorp.shape}")
print(f"Attrition rate: {techcorp['left_within_6mo'].mean():.1%}")
print("Categorical columns: ['department', 'job_level', 'remote_status', 'manager_id']")

**Reading the output:**

The dataframe prints 2,000 rows. Three of the columns (`department`, `job_level`, `remote_status`) show up as `object` dtype — pandas' way of saying *"text labels, not numbers."* That is the moment this course has been working toward: every nb01–nb08 dataset was entirely numeric, so you could get away with `StandardScaler` and nothing else. TechCorp breaks that pattern. Feeding an `object`-typed column straight into `LogisticRegression` would raise a "could not convert string to float" error — which is the problem `ColumnTransformer` solves by routing numeric and categorical columns to different preprocessing branches.

The attrition rate prints at roughly **38%** (the synthetic generator is mildly aggressive on purpose, so the positive class has enough mass for stable CV). That is more imbalanced than Breast Cancer (63/37) but less imbalanced than a typical fraud or churn dataset (95/5). You now know exactly how to handle it — keep `stratify=y` in every split, and watch both ROC-AUC (threshold-free, our default) and the confusion matrix at your chosen threshold.

The 30 `engagement_metric_*` columns are pure noise, drawn from a standard normal. They are here because real HR data really does export dozens of low-signal telemetry fields — and they make the selection-bias leak in Exercise 2 clearly visible. The signal-to-noise ratio in the feature space is exactly what makes `SelectKBest`-outside-the-pipeline a dangerous default.

> **A question that often comes up here:** *"The attrition rate is \~38%, not the 22% I was expecting from the Gemini prompt."* The generator's logit weights push a bit higher than the target rate in this seed — which is still fine for our purposes because the dataset stays solidly balanced enough for stratified CV to work cleanly. Class balance matters less than class *presence in every fold*, which is what `StratifiedKFold` guarantees.

---

### 2.2 ColumnTransformer — mixing numeric and categorical preprocessing

You have a dataset with two kinds of columns — numeric (five core + 30 noise) and categorical (four columns, one of them high-cardinality). Numeric columns want `StandardScaler` (they need centering and scaling so LogReg's coefficients have comparable magnitudes). Categorical columns want `OneHotEncoder` (they cannot be fed to a linear model as strings). You need **both**, and you need both to be applied *inside the same pipeline* so that every CV fold refits them on its own training portion only.

`ColumnTransformer` is the sklearn object that lets you do this. Think of it as two parallel assembly lines that start with the same raw DataFrame and end by concatenating their outputs into one wide feature matrix:

```
                    [X raw DataFrame]
                          |
              +-----------+-----------+
              |                       |
         numeric branch          categorical branch
         StandardScaler          OneHotEncoder
         on NUM_COLS             on CAT_COLS
              |                       |
              +-----------+-----------+
                          |
                 [X preprocessed, concatenated]
                          |
                  LogisticRegression
```

In code, this is one constructor call:

```python
preprocess = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)
])
```

Each tuple is `(name, transformer, list_of_columns)`. The `name` is just a label sklearn uses in `cv_results_` (e.g., `num__with_mean` if you later tune it). The transformer is the step that branch applies. The list of columns tells `ColumnTransformer` which columns of the incoming DataFrame flow into that branch. `NUM_COLS` and `CAT_COLS` are disjoint — no column goes through both branches — so the outputs concatenate cleanly.

Then you wrap the whole thing in a `Pipeline`:

```python
tc_pipeline = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
```

Now `cross_val_score(tc_pipeline, X_train, y_train, cv=cv_tc, scoring='roc_auc')` does the right thing on every fold: the scaler's means are learned from each fold's training portion only, the one-hot vocabulary is learned from each fold's training portion only, and none of the held-out-fold data contaminates the fit. That is what "leak-free" means in concrete terms.

**The `handle_unknown='ignore'` flag is non-negotiable.** If a prediction-time row shows a category that was never in the training portion of the fold — a new hire in a brand-new department, a manager hired after your data snapshot — the encoder has two choices: raise an exception (default) or emit a row of zeros for that category block (with `handle_unknown='ignore'`). The second option is the only one compatible with production deployment, where you cannot predict the full set of categories in advance.

> 💡 **Gemini Prompt:** "Split TechCorp into X (features) and y (target `left_within_6mo`). Apply a stratified 60/20/20 train/val/test split with `random_state=RANDOM_SEED`. Build a `ColumnTransformer` with two branches: (a) `StandardScaler` on the 5 core numeric columns plus the 30 engagement noise columns, (b) `OneHotEncoder(handle_unknown='ignore')` on the 4 categorical columns. Wrap it in a `Pipeline` together with `LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)`. Run 5-fold stratified `cross_val_score` on `X_train`, `y_train` with `scoring='roc_auc'`. Print the fold scores, mean, SD, and 95% CI using Student's t. Label the result `TechCorp baseline — leak-free pipeline`."


In [ ]:
# --- Stratified 60/20/20 split ---
CORE_NUM_COLS = ['years_at_company', 'satisfaction_score', 'salary_pct_of_market',
                 'projects_completed_last_year', 'manager_interactions_last_quarter']
NUM_COLS = CORE_NUM_COLS + NOISE_COLS  # feed all numeric columns into the pipeline
CAT_COLS_LOW_CARD = ['department', 'job_level', 'remote_status']
CAT_COLS = CAT_COLS_LOW_CARD + ['manager_id']  # manager_id is high-cardinality (80 levels)

X_tc = techcorp[NUM_COLS + CAT_COLS]
y_tc = techcorp['left_within_6mo']

X_tc_temp, X_tc_test, y_tc_temp, y_tc_test = train_test_split(
    X_tc, y_tc, test_size=0.20, random_state=RANDOM_SEED, stratify=y_tc
)
X_tc_train, X_tc_val, y_tc_train, y_tc_val = train_test_split(
    X_tc_temp, y_tc_temp, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_tc_temp
)
print(f'Train: {len(X_tc_train)} | Val: {len(X_tc_val)} | Test: {len(X_tc_test)} (locked)')

# --- ColumnTransformer + LogReg pipeline ---
preprocess = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)
])
tc_pipeline = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])

# --- 5-fold stratified CV with CI ---
cv_tc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores_tc = cross_val_score(tc_pipeline, X_tc_train, y_tc_train,
                               cv=cv_tc, scoring='roc_auc')

mean_tc = cv_scores_tc.mean()
sd_tc = cv_scores_tc.std(ddof=1)
half_w_tc = t_crit * sd_tc / np.sqrt(k)

print('\nTechCorp baseline — leak-free pipeline')
print(f'Fold AUC: {np.round(cv_scores_tc, 4)}')
print(f'Mean AUC: {mean_tc:.4f}')
print(f'Std (k-1): {sd_tc:.4f}')
print(f'95% CI:   [{mean_tc - half_w_tc:.4f}, {mean_tc + half_w_tc:.4f}]')

**Reading the output:**

The TechCorp baseline pipeline prints five fold scores and the familiar mean ± 95% CI. Expect a mean ROC-AUC in the low- to mid-0.8s — not as sharp as MedScreen's 0.99, because HR data is genuinely noisier than medical imaging, but solidly better than the "managers guessing" baseline (ROC-AUC = 0.50 by definition, the coin-flip ceiling).

The crucial property of this pipeline is that **every statistic — standardization means, one-hot vocabularies — is computed inside each CV fold's training portion only**. Nothing the pipeline learns about column means or category levels is influenced by the held-out fold. That is what makes this number trustworthy. And that is the property Sections 2.3 and Exercise 2 are about to violate on purpose, so you can see the inflated score that results.

Two practical notes on what just happened under the hood of the one `ColumnTransformer` call:

- The **numeric branch** applied `StandardScaler` to 35 columns (5 core + 30 noise engagement metrics). Every fold sees its own mean and SD, not the full dataset's.
- The **categorical branch** applied `OneHotEncoder(handle_unknown='ignore')` to 4 columns — including `manager_id` with its 80 levels. That alone adds 80 sparse indicator columns. LogReg with default L2 handles this cheerfully. If you had 8,000 managers instead of 80, you would want either a low-cardinality re-bucketing or a properly-wrapped target encoder — not raw OHE.

> **A question that often comes up here:** *"Does the order of preprocessing inside `ColumnTransformer` matter?"* For `StandardScaler` on numerics and `OneHotEncoder` on categoricals, no — the two branches run in parallel on disjoint columns and the outputs are concatenated. Order matters *within* a branch (e.g., impute before scale, not after), and order matters in the outer `Pipeline` (preprocess before model). Inside `ColumnTransformer`'s branches, the order of the list of `(name, transformer, cols)` tuples is cosmetic only.

---

### 2.3 The intern's leaky pipeline — target encoding on full data

Now the dramatic moment of the notebook. While you were out of the office, an intern on the People Analytics team shipped a "first cut" of the TechCorp model. Their CV ROC-AUC came out *notably* higher than the baseline you just computed. They were celebrating. The CHRO wants to see the numbers at the Monday meeting.

The intern reached for a Kaggle trick they had read about: **target encoding**. Before you dive into what went wrong, let us spend a minute on what target encoding *is*, because many undergrads meet it for the first time today.

### What target encoding is (and why it is tempting)

Suppose you have a categorical column like `manager_id` with 80 levels. The obvious encoding is one-hot: create 80 indicator columns, one per manager, each a 0/1. That works, but it has two practical downsides. It produces a **wide, sparse feature matrix** (80 mostly-zero columns), and it treats every manager as equally unknown — the model has to learn a separate coefficient for each manager from scratch, with only \~25 employees per manager to learn from. That is a lot of parameters for a little bit of signal.

**Target encoding** is the shortcut. Instead of one-hot encoding the categorical column into 80 indicators, replace each category with the **mean of the target within that category**. So for `manager_id = MGR_017`, you compute the fraction of employees under MGR_017 who resigned, and that fraction becomes the encoded value. 80 levels collapse into **one dense numeric column** that already contains the signal "how attrition-prone are this manager's employees?". In practice, on Kaggle, target encoding can give a real boost on high-cardinality categoricals — *when it is done right*.

The intern applied it to all four categorical columns: `department`, `job_level`, `remote_status`, and `manager_id`. Here is their code:

```python
# Intern's approach
techcorp_te = techcorp.copy()
for col in CAT_COLS:  # includes manager_id
    techcorp_te[col + '_te'] = (
        techcorp_te.groupby(col)['left_within_6mo'].transform('mean')
    )
    techcorp_te = techcorp_te.drop(columns=[col])
```

### Why this leaks (the mechanical reason)

Read the first line of the loop very carefully: `techcorp_te.groupby(col)['left_within_6mo'].transform('mean')`. That groups the **full dataset** by the categorical column and computes each group's mean of the **target**, then assigns the group-mean back to every row in that group.

*Every row's label is used in the computation.* Training rows, validation rows, test rows — all of them contribute to each group's mean. So the encoded column `manager_id_te` for a specific employee contains the average resignation rate for that employee's manager, computed using — among others — *that employee's own label*.

Then the intern does a train/test split and runs 5-fold CV on the encoded matrix. Every CV fold's validation set was already used to compute the encoded values in the training portion. The classifier is training on a feature that carries information about the validation rows' labels.

### Why it bites so hard on `manager_id`

This is the part that often feels most counterintuitive. On a low-cardinality categorical like `department` (5 levels, \~400 rows per level), the group means are averages of many rows. One training row's label barely moves the group mean. The leak is still there but it is small.

On `manager_id` with 80 levels and \~25 employees per level, **each group mean is an average of only \~25 labels**. A single training row's label shifts the group mean by roughly 1/25 = 4%. When CV later holds out 5 of those 25 rows as the validation fold, the encoded value for those 5 rows is still carrying information about their own labels — the training rows' encoded values were computed *with those labels in the mean*. The encoded column is nearly a direct readout of the target for the rows you are about to predict.

The smaller the group size, the more severe the leak. `manager_id` with 25-row groups is small enough that the leak is dramatic and easy to see.

### The cell below

Run the intern's code as they wrote it, then compare the resulting CV ROC-AUC to the leak-free Section 2.2 baseline. Expect the leaky version's mean ROC-AUC to come in several hundredths higher than the honest baseline. That gap is the *size of the lie*: the number the intern's cell prints is not a generalization estimate, it is a partial memorization of the training labels. The leak-free version is what you would actually see on new employees in production.

In [ ]:
# --- Intern's leaky pipeline: target encoding on the FULL dataset ---
techcorp_te = techcorp.copy()
for col in CAT_COLS:  # includes manager_id (80 levels)
    techcorp_te[col + '_te'] = (
        techcorp_te.groupby(col)['left_within_6mo'].transform('mean')
    )
    techcorp_te = techcorp_te.drop(columns=[col])

X_te = techcorp_te.drop(columns=['left_within_6mo'])
y_te = techcorp_te['left_within_6mo']

Xte_temp, Xte_test, yte_temp, yte_test = train_test_split(
    X_te, y_te, test_size=0.20, random_state=RANDOM_SEED, stratify=y_te
)
Xte_train, Xte_val, yte_train, yte_val = train_test_split(
    Xte_temp, yte_temp, test_size=0.25,
    random_state=RANDOM_SEED, stratify=yte_temp
)

leaky_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
cv_leaky = cross_val_score(leaky_pipeline, Xte_train, yte_train,
                           cv=cv_tc, scoring='roc_auc')
print("Intern's leaky pipeline (target encoding fit on FULL data):")
print(f'  Fold AUC: {np.round(cv_leaky, 4)}')
print(f'  Mean AUC: {cv_leaky.mean():.4f}  ← inflated\n')

# --- Leak-free baseline from Section 2.2 (OHE of every categorical, no target encoding) ---
print('Leak-free baseline from Section 2.2 (OHE only, no target encoding):')
print(f'  Fold AUC: {np.round(cv_scores_tc, 4)}')
print(f'  Mean AUC: {mean_tc:.4f}  ← realistic')

delta_leak = cv_leaky.mean() - mean_tc
print(f'\n→ The leak inflated mean ROC-AUC by {delta_leak:+.4f}.')
print('  Target encoding on the full dataset hands the model the answer, in a single column.')

**Reading the output:**

Two side-by-side CV runs on the same TechCorp dataset, and the numbers are doing exactly what the leak theory predicts.

**The leaky run.** Mean ROC-AUC lands in the low 0.83s (your exact number may shift by \~0.001 depending on how sklearn shuffles). The intern's celebration is not unfounded from their perspective — compared to the Section 2.2 baseline of roughly 0.78, the leaky pipeline posted a 5-point lift, which would be an enormous improvement if it were real.

**The leak-free Section 2.2 baseline.** Mean ROC-AUC in the high 0.77s / low 0.78s. This is the honest number — what the model would actually produce on a new TechCorp employee in production, because the pipeline only saw each fold's training portion during its scaling and encoding fits.

**The gap between them is the *size of the lie*.** A 4–5 percentage-point AUC gap on a classification task is not a rounding error; it is the difference between *"our pilot model flagged 80% of resignations"* and *"our pilot model flagged 75% of resignations"*, which in dollar terms (at USD 75k per prevented resignation) is a real amount of money. If the intern shipped the leaky pipeline to the Monday meeting without catching the leak, the CHRO would budget for the 80% number, deploy the model, and watch it underperform by that same gap in production. That is how leakage kills careers.

### The fix

**Delete the target-encoded columns and fall back to `OneHotEncoder(handle_unknown='ignore')` inside the `Pipeline`** — exactly what the Section 2.2 baseline already does. OHE never touches `y`, so it is *leak-safe by construction*. The "real" fix — target encoding that refits its per-category means on every CV fold's training portion only — requires a custom transformer (the library `category_encoders` has one called `TargetEncoder` that integrates with sklearn's CV correctly). That is a topic for later in your career; for this course, OHE is the safe default.

### The rule to internalize right now

**If a feature is computed using any row's target, the only safe way to include it is a transformer that refits per CV fold.** `OneHotEncoder` never reads `y`, so it is leak-safe. Target encoders *do* read `y`, so they must live inside the Pipeline and refit per fold. The intern's version failed because the encoding computation ran once on the full DataFrame — outside the fold loop, outside the Pipeline, before the split.

> **A question that often comes up here:** *"If target encoding is so dangerous, why do Kaggle winners use it?"* They use a version that refits per fold — properly implemented, it can give a real boost on high-cardinality categoricals. The dangerous version is the one you just saw (`groupby.transform('mean')` on the full DataFrame), which is how target encoding is often demo'd in quick tutorials and how many of you will first encounter it. The difference between "Kaggle winner move" and "public-leaderboard disaster" is whether the encoder lives inside the `Pipeline` that `cross_val_score` or `GridSearchCV` evaluates. This is not a small distinction; it is the entire game.

---

### 2.4 `FunctionTransformer` — domain features without leakage

Stakeholders almost always want a feature that is not in the raw data. At TechCorp, the HR Business Partner (HRBP — the person who actually talks to employees and managers) is convinced that *ratios matter more than absolute values*: an employee whose `salary_pct_of_market` is 85 is more attrition-prone than one at 130 (they know they are underpaid), and an engineer with 20 manager interactions across 3 projects gets more attention per unit of work than one with 5 interactions across 10 projects.

The HRBP's specific ask: add an `interactions_per_project` ratio defined as `manager_interactions_last_quarter / (projects_completed_last_year + 1)`. (The `+ 1` is a "Laplace smoothing" hack to avoid dividing by zero for employees who completed no projects — a small but real gotcha in any ratio feature.)

### The amateur move vs. the professional move

**The amateur move:** compute the ratio in a pandas `assign` call before splitting, treat the new column as raw data, and feed it to the existing pipeline. On this specific ratio — a pure arithmetic combination of two raw columns, with no dependence on any training statistic — the amateur move actually works, because there is nothing to leak. No mean, no SD, no group-level statistic. A simple function of two existing columns.

**The professional move:** use `FunctionTransformer` to wrap the ratio computation as a pipeline step anyway. This makes the ratio compute fold-by-fold inside the CV loop, and more importantly, it *establishes the habit*: every feature-engineering step is a pipeline step, no exceptions. The habit matters because the moment you add a feature that depends on a training statistic — say, `salary_pct_of_dept_median` (each employee's salary as a fraction of their department's median) — the "compute it in pandas before splitting" approach silently leaks: the department median would be computed using validation and test rows.

The one-line rule: **if you are even slightly unsure whether a feature engineering step leaks, put it inside the pipeline**. `FunctionTransformer` exists so that rule costs you nothing.

### How it works mechanically

`FunctionTransformer(func, validate=False)` wraps an arbitrary Python function `func` into a sklearn transformer that can be slotted into a `Pipeline`. On every CV fold's training portion, sklearn calls `func(X_train_fold)` to transform the training data; on the validation portion, it calls `func(X_val_fold)`. If `func` is a pure function (no training statistics, no global state), both calls produce the same transformation and nothing can leak. If `func` were to depend on a statistic of its input, sklearn would recompute that statistic per fold — which is exactly the leak-free property you want.

**The test cell below** adds the HRBP's `interactions_per_project` ratio via `FunctionTransformer`, reruns 5-fold CV, and compares the new CI to the Section 2.2 baseline CI. Same decision rule as every model comparison today: **does the ratio feature's CI clear the baseline's CI?** If yes, the feature is earning its keep; add it to the champion. If no (CIs overlap), the HRBP's hypothesis is plausible but unproven on this sample, and the simpler model (no ratio) is the defensible choice.

> 💡 **Gemini Prompt:** "Write a function `add_interactions_per_project(X)` that takes a pandas DataFrame with `manager_interactions_last_quarter` and `projects_completed_last_year`, returns a NumPy array with an added column `manager_interactions_last_quarter / (projects_completed_last_year + 1)`. Wrap it in a `FunctionTransformer(validate=False)`. Insert it as the first step in the pipeline (before the `ColumnTransformer`) and re-run 5-fold stratified CV on `X_tc_train`. Report the mean ROC-AUC and 95% CI, and compare to the baseline from Section 2.2."


In [ ]:
# --- FunctionTransformer to add interactions-per-project ratio ---
def add_interactions_per_project(X):
    X = X.copy()
    X['interactions_per_project'] = (
        X['manager_interactions_last_quarter'] /
        (X['projects_completed_last_year'] + 1)
    )
    return X

add_ratio = FunctionTransformer(add_interactions_per_project, validate=False)

NUM_COLS_RATIO = NUM_COLS + ['interactions_per_project']
preprocess_ratio = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS_RATIO),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)
])

tc_pipeline_ratio = Pipeline([
    ('add_ratio', add_ratio),
    ('prep', preprocess_ratio),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])

cv_tc_ratio = cross_val_score(tc_pipeline_ratio, X_tc_train, y_tc_train,
                              cv=cv_tc, scoring='roc_auc')

mean_ratio = cv_tc_ratio.mean()
sd_ratio = cv_tc_ratio.std(ddof=1)
half_w_ratio = t_crit * sd_ratio / np.sqrt(k)

print('TechCorp pipeline + FunctionTransformer(interactions_per_project)')
print(f'Mean AUC: {mean_ratio:.4f}')
print(f'95% CI:   [{mean_ratio - half_w_ratio:.4f}, {mean_ratio + half_w_ratio:.4f}]')

print('\nCompare to Section 2.2 baseline:')
print(f'  Baseline mean AUC: {mean_tc:.4f}  (CI [{mean_tc - half_w_tc:.4f}, {mean_tc + half_w_tc:.4f}])')
print(f'  +Ratio   mean AUC: {mean_ratio:.4f}  (CI [{mean_ratio - half_w_ratio:.4f}, {mean_ratio + half_w_ratio:.4f}])')

overlap_fe = not (mean_tc + half_w_tc < mean_ratio - half_w_ratio or
                  mean_ratio + half_w_ratio < mean_tc - half_w_tc)
print(f"\n→ CIs {'OVERLAP' if overlap_fe else 'do NOT overlap'}: "
      f"the ratio feature {'does not convincingly help' if overlap_fe else 'is a real edge'}.")

**Reading the output:**

Two pipelines, two CIs. The question is the same as every exercise from nb08 forward: *does the engineered ratio meaningfully improve the model?* And the answer comes from the CI-overlap rule, not from comparing point estimates.

If the two CIs overlap, the engineered ratio is *not* a statistically convincing lift — the HRBP's hypothesis is plausible but unproven on this sample, and you should leave the ratio out of the reported champion unless a separate argument (interpretability, stakeholder buy-in) keeps it in. If the CIs do not overlap, you have evidence that the ratio matters — quote it in the HR readout.

The pedagogical point is not whether the ratio helps on *this* synthetic dataset; it is the discipline: **every feature addition is tested with CV + CI overlap**, the same way nb08 tested Ridge vs OLS and $C=1.0$ vs $C=0.01$. You now have one tool (`FunctionTransformer`) that lets you add arbitrary domain features to a pipeline without leaking — which is exactly what you need to translate a stakeholder's hypothesis into a testable feature.

> **A question that often comes up here:** *"If `FunctionTransformer` doesn't learn anything from the data, why does it need to be inside the `Pipeline`?"* On a strictly static ratio like `manager_interactions / (projects + 1)`, it does not strictly need to be. But the *habit* matters: as soon as you add a feature that depends on a training statistic (a training-set mean, a training-set quantile, a training-set category frequency), the outside-the-pipeline version would silently leak. The professional default is *"every feature-engineering step is a pipeline step,"* without exceptions. That single rule makes it impossible to forget a leak on a day when you are tired or in a rush.

---

## 📝 PAUSE-AND-DO Exercise 2 (10 minutes)

**Task:** Find and fix a second leak — this one a different pattern from Section 2.3's target encoding.

Section 2.3 walked you through a dramatic leak (target encoding on the full dataset). This exercise shows you a subtler one: a feature selector applied *before* the split instead of inside the pipeline. The inflation on this data will be smaller than the target-encoding case — a few thousandths to a few hundredths of a point rather than 0.03 — but the principle is identical and the fix is identical. Small leaks matter too: a 0.005 inflation can change a CI-overlap verdict, which can change which model ships.

Another teammate sends you the snippet below. They have heard about leakage and are careful — they are **not** doing target encoding. Instead they are using a feature selector to reduce the dimensionality of the one-hot-encoded feature matrix (pruning down from 100+ OHE columns to only the ones that correlate with attrition). Their CV ROC-AUC came out a bit higher than your leak-free baseline. Read the code carefully before running.

```python
# Teammate's pipeline with univariate feature selection
X_all_dummies = pd.get_dummies(techcorp.drop(columns=['left_within_6mo']))
y_all = techcorp['left_within_6mo']

# Select the top-8 features by univariate correlation with the target — on ALL rows
selector = SelectKBest(f_classif, k=8)
X_selected = selector.fit_transform(X_all_dummies, y_all)

# Now split and CV on the pre-selected matrix
Xs_temp, Xs_test, ys_temp, ys_test = train_test_split(
    X_selected, y_all, test_size=0.20, random_state=RANDOM_SEED, stratify=y_all
)
Xs_train, Xs_val, ys_train, ys_val = train_test_split(
    Xs_temp, ys_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=ys_temp
)

pipe_sel = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
cv_sel = cross_val_score(pipe_sel, Xs_train, ys_train, cv=cv_tc, scoring='roc_auc')
print(f'Pre-selected mean AUC: {cv_sel.mean():.4f}')
```

**Your job:**

1. Run the teammate's code and record the leaky mean.
2. Identify the leak in one sentence. *Hint:* `SelectKBest.fit(X, y)` reads `y` — does it do so on training rows only, or on every row?
3. Write a leak-free version by putting `SelectKBest` **inside** a `Pipeline` (chain it after the `ColumnTransformer` from Section 2.2, before the classifier). Run 5-fold stratified CV on `X_tc_train`, `y_tc_train`.
4. Print a final line: `Leak inflated mean ROC-AUC by +<delta>`.
5. Note: even if the inflation on this synthetic data is small, the *principle* is identical to Section 2.3. The Kaggle leaderboard does not reward "small" leaks; it punishes every leak equally when the private test scores reveal.

> **A question that often comes up here:** *"If the leak inflation is only 0.005, does it really matter?"* For a stakeholder's confidence interval — yes, because 0.005 can easily be the difference between "CIs overlap" and "CIs do not overlap" on a comparison you care about. More importantly: leaks you tolerate on small data scale catastrophically on big data. The pattern of *"fit outside the pipeline"* is always wrong. Fixing it when the stakes are low (here, 10 minutes in a classroom) is cheaper than discovering the same pattern on a Kaggle private leaderboard at midnight.

---

> 💡 **Gemini Prompt:** "First, reproduce the teammate's leaky pipeline exactly as written: one-hot-encode the full TechCorp dataset with `pd.get_dummies(techcorp.drop(columns=['left_within_6mo']))`, fit `SelectKBest(f_classif, k=8)` on the entire matrix and target, then split the resulting selected matrix 60/20/20 with `random_state=RANDOM_SEED` and `stratify=y` on both calls. Build a `Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))])`, run `cross_val_score` on the training portion with `cv=cv_tc` and `scoring='roc_auc'`, and store `mean_leaky = scores.mean()`. Then build a leak-free pipeline that puts every data-derived step *inside* a single `Pipeline`: `ColumnTransformer([('num', StandardScaler(), NUM_COLS), ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)])` → `SelectKBest(f_classif, k=8)` → `LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)`. Run `cross_val_score` on `X_tc_train`, `y_tc_train` with `cv=cv_tc` and `scoring='roc_auc'` and store `mean_fixed = scores.mean()`. Print both means with 4-decimal precision and the delta `mean_leaky - mean_fixed`. End with a single printed sentence naming where the leak is in the teammate's code."
>
> **After running, verify:**
> - The leaky-pipeline mean ROC-AUC prints first (slightly higher)
> - The leak-free mean ROC-AUC prints next (slightly lower — typically by 0.005 to 0.01 on this data)
> - The delta `mean_leaky - mean_fixed` is positive and equals the inflation the leak buys
> - The final printed sentence identifies the leak as `SelectKBest.fit(X, y)` running on the full dataset *before* the train/val/test split
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Reproduce the teammate's pre-select-then-split pipeline and record cv_sel.mean()
#   2. Build a leak-free pipeline: ColumnTransformer (Section 2.2) → SelectKBest(k=8) → LogisticRegression
#   3. Run cross_val_score on X_tc_train, y_tc_train with cv=cv_tc and scoring='roc_auc'
#   4. Print both means and the delta
#   5. In the analysis cell below, explain WHERE the leak is (one sentence)


### YOUR ANALYSIS:

**Question 1:** In one sentence, where is the leak in the teammate's code?  
[Your answer]

**Question 2:** By how much did the leak inflate the mean ROC-AUC? Is that inflation large enough to change the verdict of a CI-overlap comparison with a rival model?  
[Your answer]

**Question 3:** What is the one-line rule you would teach this teammate so this never happens again?  
[Your answer]

---

## 3. Toolkit Recap — What You Hold After nb01–nb09

This section is your one-page reference for the full mid-course toolkit. Skim it now; treat it as the cheat sheet for any new business case you encounter (*"what target? what metric? what split? where is the leakage risk?"*). Nothing below is new — every item lives in a notebook you have already worked through. The point of the recap is to lay the toolkit out in one place so future cases hit a prepared mind, not a scattered one.

### 3.1 Concepts — the mental models the toolkit rests on

The course's core *concepts* are not sklearn methods; they are the language any business-case prompt uses. Each is introduced explicitly in a specific notebook and is part of the toolkit you will reach for from here on.

- **The statistical-learning frame ($Y = f(X) + \epsilon$, nb01).** Every problem you have seen reduces to estimating $f$. Two flavors: **regression** (continuous $Y$, like house prices in nb01–nb05) and **classification** (categorical $Y$, like cancer or attrition in nb06–nb09). Knowing which flavor a business case calls for is usually the first decision you have to make.
- **Bias–variance tradeoff (nb01, nb04, nb05).** Bias is how wrong the model is on average; variance is how much its predictions move when the training sample changes. High bias → underfitting; high variance → overfitting. Regularization (nb05) is *literally* a knob that buys lower variance at the cost of a sliver of bias.
- **Overfitting vs underfitting (nb04).** Diagnose by comparing train and validation scores: train ≫ validation = overfitting; both equally bad = underfitting. nb04's polynomial blow-up was the canonical example; nb05's Ridge/Lasso was the canonical fix.
- **Curse of dimensionality (nb01, nb04).** As the feature count grows, every training point ends up alone in a sparsely populated neighborhood and naive distance-based reasoning falls apart. This is *why* regularization matters as soon as you add interactions or polynomials.
- **Data leakage (nb01, nb02, nb09).** Two flavors. **Target leakage**: a feature is computed using information that would not be available at prediction time (the target's own value, future timestamps, post-outcome columns). **Train–test contamination**: any `.fit(...)` step that touches held-out data — scaling, encoding, target encoding, feature selection — *before* the split. nb09's two case studies (target encoding on the full dataset and `SelectKBest` outside the pipeline) are the canonical traps you should be able to spot in code.

### 3.2 Workflow — the four moves that go in every analysis

Every notebook from nb01 onward executed the same four-move sequence. Apply it to any new case the same way.

1. **EDA + split (nb01, nb06).** Audit dtypes, missingness, target distribution; then a stratified 60/20/20 train/val/test split with `random_state=RANDOM_SEED`. Stratify on classification problems. The test set is locked until nb14.
2. **Build the pipeline (nb02, nb09).** `ColumnTransformer` for column-type-specific preprocessing — numeric → `StandardScaler`; categorical → `OneHotEncoder(handle_unknown='ignore')` — then your model, all wrapped in a single `Pipeline`. Every data-derived step lives *inside* the pipeline so CV refits each fold cleanly.
3. **Evaluate honestly (nb03, nb07, nb08).** Pick a metric the business cares about (MAE / RMSE / $R^2$ for regression; precision / recall / F1 / ROC-AUC for classification, with thresholding when costs are asymmetric). Run 5-fold CV on `X_train`, never on `X_test`. Report mean ± 95% CI using Student's $t$ critical value at $k-1$ degrees of freedom.
4. **Compare with the CI-overlap rule (nb08, nb09).** When two candidates' 95% CIs overlap, the choice is *not* performance-driven — pick the simpler model (more regularization, smaller `C`, fewer features). When CIs do not overlap, the top one wins outright. *This decision primitive is the spine of the rest of the course.*

### 3.3 Tools — the sklearn primitives at your fingertips

| Layer | Primitive | Where it lives in the course |
|---|---|---|
| Preprocessing | `StandardScaler`, `OneHotEncoder(handle_unknown='ignore')`, `FunctionTransformer` | nb02, nb09 |
| Composition | `Pipeline`, `ColumnTransformer` | nb02, nb09 |
| Regression | `LinearRegression`, `Ridge`, `Lasso` | nb03–nb05 |
| Classification | `LogisticRegression` (`C` is regularization, `class_weight='balanced'` for imbalance) | nb06, nb07 |
| Splitting | `train_test_split(stratify=y)`, `KFold`, `StratifiedKFold` | nb01, nb06, nb08 |
| Evaluation | `cross_val_score`, `cross_val_predict`, regression metrics (`mean_absolute_error`, `mean_squared_error`, `r2_score`), classification metrics (`roc_auc_score`, `classification_report`, `confusion_matrix`) | nb03, nb07, nb08 |
| Tuning | `GridSearchCV`, `RandomizedSearchCV` (read `cv_results_`, apply CI-overlap) | nb09 |

### 3.4 Decision rules — the heuristics you now apply

These are the one-line rules you should have memorized walking into any new analysis. They apply to every business case from here on.

- **Scale when the model uses distances or coefficients.** Logistic regression, Ridge, Lasso → scale. Trees and tree ensembles (Week 3) → no scaling needed.
- **Stratify when classes are imbalanced.** Classification splits should be stratified, and so should `KFold` (use `StratifiedKFold`).
- **Metric choice flows from cost asymmetry.** If false negatives are expensive and false positives are cheap, optimize recall; if the opposite, optimize precision. F1 balances both. ROC-AUC is metric-agnostic and good for ranking; PR-AUC is better when the positive class is rare. A typical business case (*"missed fraud costs USD 50,000; a false alarm costs USD 50"* — pick the metric) makes the cost asymmetry concrete.
- **Ridge keeps everything small; Lasso zeros things out.** Ridge for stable coefficient estimates with all features kept; Lasso for automatic feature selection (sparsity).
- **`C` is the *inverse* of regularization strength** (LogReg). Small `C` (e.g., 0.01) → strong regularization → simpler model. Large `C` (e.g., 100) → weak regularization → flexible boundary. `C = 1.0` is sklearn's default.
- **CI-overlap rule (the course's spine).** Tied 95% CIs → pick the simpler model. Non-overlapping CIs → top wins outright. Never report a single point estimate without a CI.
- **Leakage rule (one line).** Any `.fit(...)` call on data — scaler, encoder, imputer, selector, target encoder — must live *inside* the `Pipeline` that `cross_val_score` or `GridSearchCV` evaluates. If a transform takes $X$ or $y$ during `fit` and lives outside, it is leaking.

> **A question that often comes up here:** *"How do I make sure I can apply this toolkit on a new case?"* Re-run nb01 → nb09 in order, paying attention to *why* each notebook exists, not just *what* code it runs. Any new business case hands you brand-new context and asks you to choose target / metric / split / pipeline / risk-flag — none of those decisions are looked up in code; they all come from the concepts and decision rules above. If you can write one paragraph for each row of the table in 3.3 explaining when you would reach for that primitive, you can wield the toolkit on whatever comes next.

---


## 4. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **`GridSearchCV` is nb08's CV ritual, run on every point in a grid.** Every row of `cv_results_` has a mean, an SD, and (with two lines of `scipy.stats`) a 95% CI. Rank by mean, then break ties toward the simpler model using the CI-overlap rule. This is the decision primitive for the rest of the course.
2. **`RandomizedSearchCV` is the same idea for larger grids.** Sample `n_iter` points from distributions (`loguniform`, `uniform`) when an exhaustive grid would take too long. The output format is identical, so everything you know about reading `cv_results_` transfers.
3. **`ColumnTransformer` is the bridge to real-world data.** Numeric columns need `StandardScaler`; categorical columns need `OneHotEncoder(handle_unknown='ignore')`. Both branches live inside one `ColumnTransformer`, which lives inside one `Pipeline`. That pattern is the template for every Kaggle submission and every final-project pipeline from here on.
4. **`FunctionTransformer` embeds domain features without leaking.** Any pure function of a DataFrame can become a pipeline step. Test every new feature with the CI-overlap rule, not a one-shot validation comparison.
5. **Leakage hides inside `fit` calls on the full dataset.** Any step that takes `X` (for scaling, selection, encoding) or `y` (for feature selection, target encoding) must live inside the `Pipeline`. If the score looks suspiciously good, check for a `.fit(...)` call outside `cross_val_score` — that is the first place to look.
6. **Both leaks in today's notebook inflated ROC-AUC by a non-trivial margin.** That gap is the *size of the lie* a leaky pipeline tells. On a Kaggle leaderboard, a leak of this size masks the real generalization gap and wrecks your final standing when the private test set is scored.

### Critical Rules:

> **"If a preprocessing step uses any data statistic, it belongs inside the `Pipeline`."**

> **"`cv_results_` is a stack of nb08 runs. Read it that way."**

> **"Break ties toward the simpler model — the CI-overlap rule is the rest of the course's decision primitive."**

### Next Steps:

- **nb11–nb13 (trees, forests, gradient boosting) raise the stakes.** Every new model class introduces more hyperparameters — `max_depth`, `min_samples_leaf`, `n_estimators`, `learning_rate` — and today's `GridSearchCV` reflex is how you will handle them without turning into a random-button-presser.
- **The Kaggle competition** (launched on Day 5, due Day 20) is the full pipeline you just built — `ColumnTransformer → FunctionTransformer → GridSearchCV → CI-overlap champion → refit on all training data → predict on test → submission.csv`. If you can run today's notebook end-to-end, you can ship a Kaggle submission. Section 2's leakage case studies are there so you do not ship a *leaky* submission.

---


## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in your code and analysis in both PAUSE-AND-DOs.
2. **Verify outputs**: Restart the runtime (`Runtime → Restart session`) and run all cells top to bottom. Every cell should execute without error.
3. **Download**: `File → Download → Download .ipynb` to save your completed notebook.
4. **Submit on Brightspace** in the Participation Assignments section.

### Grading:

- Exercise 1 complete with CI-overlap verdict: 50%
- Exercise 2 complete with leak diagnosis: 50%

---

## 5. Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* (ISLP), Ch. 5 (Resampling Methods) and Ch. 6 (Linear Model Selection and Regularization). Springer.
- Provost, F., & Fawcett, T. (2013). *Data Science for Business*. Chapters on overfitting, evaluation, and pipeline discipline.
- scikit-learn User Guide — [Tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)
- scikit-learn User Guide — [ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data)
- scikit-learn User Guide — [Common pitfalls in interpretation and evaluation](https://scikit-learn.org/stable/common_pitfalls.html)
- Kaufman, S., Rosset, S., & Perlich, C. (2012). *Leakage in Data Mining: Formulation, Detection, and Avoidance.*

---

<center>

Thank you!

</center>